# Information
This script applies fundamental ML to the data at hand in order to analyze the relations between the variables of interest from a different perspective. Specifically, The goal is to predict infant age based on the composition and optional additional metadatd

# 0. Setup

load required packages
define paths

In [18]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_predict
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import qiime2 as q2

%matplotlib inline

In [19]:
data_dir="../../data"
featuretable_dir="../../output/02_taxonomicclassification"
output_dir="../../output/05_ML"

In [20]:
! mkdir -p $output_dir 

# 1. data import

In [55]:
meta = pd.read_csv(f"{data_dir}/metadata.tsv", sep="\t", index_col="sample-id")
meta.head()

,collection_date,country,instrument,sample_number,delivery_mode,host_sex,diet_milk,city,latitude,longitude,urca_code,urca_description,host_id,host_age,seq_depth,age_months
sample-id,,,,,,,,,,,,,,,,
SRR25052523,19.09.2018,usa,illumina novaseq 6000,1,csection,female,NaN,falls church,38.86,-77.23,9,<1 hour to large city (1 - 5 million),10,2,17325587,2.0
SRR25052512,20.12.2018,usa,illumina novaseq 6000,2,csection,female,mixed,falls church,38.86,-77.23,9,<1 hour to large city (1 - 5 million),10,6,12407450,6.0
SRR25052501,08.07.2019,usa,illumina novaseq 6000,3,csection,female,mixed,falls church,38.86,-77.23,9,<1 hour to large city (1 - 5 million),10,12,11964978,12.0
SRR25052458,30.06.2020,usa,illumina novaseq 6000,4,csection,female,mixed,falls church,38.86,-77.23,9,<1 hour to large city (1 - 5 million),10,24,10921526,24.0
SRR25052720,28.08.2018,usa,illumina novaseq 6000,5,vaginal,male,NaN,falls church,38.86,-77.23,9,<1 hour to large city (1 - 5 million),105,0,12988703,0.0


# 2. baseline regressor

In [22]:
! qiime sample-classifier regress-samples \
  --i-table $featuretable_dir/feature-table.qza \
  --m-metadata-file $data_dir/metadata.tsv \
  --m-metadata-column host_age \
  --p-test-size 0.2 \
  --p-estimator RandomForestRegressor \
  --p-random-state 22 \
  --output-dir $output_dir/baseline

Saved SampleEstimator[Regressor] to: ../../output/05_ML/baseline/sample_estimator.qza
Saved FeatureData[Importance] to: ../../output/05_ML/baseline/feature_importance.qza
Saved SampleData[RegressorPredictions] to: ../../output/05_ML/baseline/predictions.qza
Saved Visualization to: ../../output/05_ML/baseline/model_summary.qzv
Saved Visualization to: ../../output/05_ML/baseline/accuracy_results.qzv


In [23]:
q2.Visualization.load(f"{output_dir}/baseline/accuracy_results.qzv")

<visualization: Visualization uuid: 0c19aab8-e900-4db6-9348-8c4c0c320377>

# 3. biological enrichment

input data: microbial composition, delivery_mode, host_sex, diet_milk

In [56]:
add_cols = ["delivery_mode", "host_sex", "diet_milk"]
M = pd.get_dummies(meta[add_cols], dtype=float)
M.to_csv(f"{output_dir}/add_cols.tsv", sep="\t", index_label="sample-id")

In [57]:
feature_table = q2.Artifact.load(f"{featuretable_dir}/feature-table.qza").view(pd.DataFrame)

In [58]:
feature_table.head() # samples x features

,100177,100225,1002546,1002672,1002689,1003110,1003114,1003177,1004,1004279,...,99653,996558,996801,99807,999,999141,999183,999883,999931,0
SRR25052448,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,299596.0
SRR25052449,0.0,1.0,2.0,0.0,2.0,3.0,0.0,0.0,3.0,1.0,...,2.0,110.0,1.0,0.0,2.0,0.0,11.0,0.0,0.0,3008240.0
SRR25052450,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,...,3.0,19.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,1704204.0
SRR25052451,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,...,0.0,17.0,0.0,1.0,13.0,0.0,0.0,0.0,0.0,1096036.0
SRR25052452,0.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,8.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,311.0,0.0,0.0,1864597.0


In [59]:
! qiime sample-classifier metatable \
  --i-table $featuretable_dir/feature-table.qza \
  --m-metadata-file $output_dir/add_cols.tsv \
  --o-converted-table $output_dir/feature-table-enriched.qza

Saved FeatureTable[Frequency] to: ../../output/05_ML/feature-table-enriched.qza


In [62]:
q2.Artifact.load(f"{output_dir}/feature-table-enriched.qza").view(pd.DataFrame).head()

,100177,100225,1002546,1002672,1002689,1003110,1003114,1003177,1004,1004279,...,999183,999883,999931,0,delivery_mode_csection,delivery_mode_vaginal,host_sex_female,host_sex_male,diet_milk_bd,diet_milk_mixed
SRR25052448,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,299596.0,0.0,1.0,1.0,0.0,0.0,0.0
SRR25052449,0.0,1.0,2.0,0.0,2.0,3.0,0.0,0.0,3.0,1.0,...,11.0,0.0,0.0,3008240.0,0.0,1.0,0.0,1.0,1.0,0.0
SRR25052450,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,...,0.0,0.0,0.0,1704204.0,0.0,1.0,0.0,1.0,1.0,0.0
SRR25052451,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,...,0.0,0.0,0.0,1096036.0,0.0,1.0,0.0,1.0,1.0,0.0
SRR25052452,0.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,8.0,0.0,...,311.0,0.0,0.0,1864597.0,0.0,1.0,0.0,1.0,1.0,0.0


In [63]:
! qiime sample-classifier regress-samples \
  --i-table $output_dir/feature-table-enriched.qza \
  --m-metadata-file $data_dir/metadata.tsv \
  --m-metadata-column host_age \
  --p-test-size 0.2 \
  --p-estimator RandomForestRegressor \
  --p-random-state 22 \
  --output-dir $output_dir/biological_enrichment

Saved SampleEstimator[Regressor] to: ../../output/05_ML/biological_enrichment/sample_estimator.qza
Saved FeatureData[Importance] to: ../../output/05_ML/biological_enrichment/feature_importance.qza
Saved SampleData[RegressorPredictions] to: ../../output/05_ML/biological_enrichment/predictions.qza
Saved Visualization to: ../../output/05_ML/biological_enrichment/model_summary.qzv
Saved Visualization to: ../../output/05_ML/biological_enrichment/accuracy_results.qzv


In [64]:
q2.Visualization.load(f"{output_dir}/biological_enrichment/accuracy_results.qzv")

<visualization: Visualization uuid: 4481f7b6-6b09-4f01-8dab-95040f4354d2>

# 4. technical enrichment

Gradient Boosting regressor, ncv train/test split

In [67]:
! qiime sample-classifier regress-samples-ncv \
  --i-table $output_dir/feature-table-enriched.qza \
  --m-metadata-file $data_dir/metadata.tsv \
  --m-metadata-column host_age \
  --p-estimator GradientBoostingRegressor \
  --p-n-estimators 20 \
  --p-random-state 22 \
  --o-predictions $output_dir/technical_enrichment_predictions.qza \
  --o-feature-importance $output_dir/technical_enrichment_featureimportance.qza

Saved SampleData[RegressorPredictions] to: ../../output/05_ML/technical_enrichment_predictions.qza
Saved FeatureData[Importance] to: ../../output/05_ML/technical_enrichment_featureimportance.qza


In [68]:
! qiime sample-classifier scatterplot \
  --i-predictions $output_dir/technical_enrichment_predictions.qza \
  --m-truth-file $data_dir/metadata.tsv \
  --m-truth-column host_age \
  --o-visualization $output_dir/technical_enrichment_predictions.qzv

Saved Visualization to: ../../output/05_ML/technical_enrichment_predictions.qzv


In [70]:
q2.Visualization.load(f"{output_dir}/technical_enrichment/technical_enrichment_predictions.qzv")

<visualization: Visualization uuid: b3713f3d-5d81-41d6-ad92-5b9299289dcb>

# 5. bias from one child having multiple measurements?
fit two baseline regressors; one specifically puts one child only into train OR test set (grouped CV) while the other one doesn't account for this potential bias-adding factor. If the latter performs much better than the first one, the model is indeed learning host-specific composition and the previously fit models are invalid. On the contrary, similar model performances indicate absence or little impact of host-id-biasing effect

In [73]:
ft = q2.Artifact.load(f"{featuretable_dir}/feature-table.qza").view(pd.DataFrame)
meta = pd.read_csv(f"{data_dir}/metadata.tsv", sep="\t", index_col="sample-id")
meta = meta.loc[ft.index]

y = meta["host_age"]
groups = meta["host_id"]   

rf = RandomForestRegressor(n_estimators=500, random_state=22, n_jobs=-1)

for name, cv in [("random KFold", KFold(5, shuffle=True, random_state=22)),
                 ("GroupKFold", GroupKFold(5))]:
    pred = cross_val_predict(rf, ft, y,
                             cv=cv,
                             groups=groups if name == "GroupKFold" else None)
    
    print(f"{name}: MAE={mean_absolute_error(y, pred):.2f}, R2={r2_score(y, pred):.2f}")

random KFold: MAE=2.97, R2=0.77
GroupKFold: MAE=2.91, R2=0.79


Almost identical performances of the model --> host id has very small effect on regression model (i.e. negligible bias, if any at all). In other words, the model trained on ungrouped CV generalizes well to unseen data

Since the baseline model (fit as very first in this script) performs almost as good as all other fit models and also yields a fairly good performance on the absolute scale (R^2 ~ 0.75, i.e. 3/4 of all variance is explained), there is not really a need to fit more rigorous / sophisticated models. Additionally, interpretability is much more straightforward in such a baseline compared to more complex models (worst case "blackbox model" NN)